# HAMIC — TPU pretraining

Choose a run configuration, inspect it, fetch the corpus, then run a small benchmark or the
real training budget. Work is published to `runs/<run_id>/` in the private Hub repository, so a
preempted TPU resumes from the latest committed checkpoint. A completed run is not trained again.

Set `CONFIG_NAME = ""` to use the default supported configuration. The token is read as
`HF_KEY` (TPU only).

In [ ]:
HAMIC_REF = "main"
HUB_REPO = "HaianCao/hamic-runs"
DATA_REPO = "HaianCao/hamic-data"
XET_HIGH_PERFORMANCE = True

# Blank = the default supported configuration.
CONFIG_NAME = ""
# "bench" runs BENCH_STEPS; "train" runs config.total_steps and resumes from the Hub.
MODE = "bench"
BENCH_STEPS = 50
BENCH_LR = ""
BENCH_WARMUP = ""

In [ ]:
import json
import os
import shutil
import sys
import tarfile
import tempfile
import urllib.request
from pathlib import Path

os.environ["HF_XET_HIGH_PERFORMANCE"] = "1" if XET_HIGH_PERFORMANCE else "0"
work_root = Path(tempfile.gettempdir()) / "hamic-tpu"
source_dir = work_root / "source"
stamp = work_root / "source-ref.json"
repo = "https://github.com/HaianCao/HAMIC-LM"
with urllib.request.urlopen(
    f"https://api.github.com/repos/HaianCao/HAMIC-LM/commits/{HAMIC_REF}",
    timeout=30,
) as response:
    commit = json.load(response)["sha"]
stamp_data = json.loads(stamp.read_text()) if stamp.is_file() else None
if not isinstance(stamp_data, dict):
    stamp_data = None
if (
    not source_dir.is_dir()
    or stamp_data is None
    or stamp_data.get("ref") != HAMIC_REF
    or stamp_data.get("commit") != commit
):
    archive = work_root / "source.tar.gz"
    work_root.mkdir(parents=True, exist_ok=True)
    urllib.request.urlretrieve(f"{repo}/archive/{commit}.tar.gz", archive)
    unpack = work_root / "unpack"
    shutil.rmtree(unpack, ignore_errors=True)
    unpack.mkdir()
    with tarfile.open(archive) as tar:
        try:
            tar.extractall(unpack, filter="data")
        except TypeError:
            tar.extractall(unpack)
    (top,) = unpack.iterdir()
    shutil.rmtree(source_dir, ignore_errors=True)
    shutil.move(str(top), source_dir)
    shutil.rmtree(unpack, ignore_errors=True)
    archive.unlink()
    stamp.write_text(json.dumps({"ref": HAMIC_REF, "commit": commit}))
sys.path.insert(0, str(source_dir / "src"))
if "hamic_jax" in sys.modules:
    raise RuntimeError(
        "An older hamic_jax is already imported. Restart the kernel and run this notebook again."
    )
import hamic_jax

if Path(hamic_jax.__file__).resolve().parent != (source_dir / "src" / "hamic_jax").resolve():
    raise RuntimeError("hamic_jax was imported from outside the pinned source tree")
print(f"Source {HAMIC_REF} resolved to {commit[:12]} · work root {work_root}")
print("The wheel excludes src/hamic_jax, so this notebook imports the pinned checkout.")

In [ ]:
from hamic_jax.dependencies import ensure_dependencies

installed = ensure_dependencies()
print("Runtime dependencies are available." if not installed else f"Installed: {', '.join(installed)}")

In [ ]:
import jax

backend = jax.default_backend()
if backend != "tpu":
    raise RuntimeError(
        f"No TPU attached (jax.default_backend()={backend!r}). Attach the TPU and rerun."
    )
devices = jax.devices()
print(
    f"JAX {jax.__version__} · backend {backend} · {len(devices)} device(s) "
    f"{devices[0].device_kind} · code {commit[:12]}"
)

In [ ]:
import getpass

hub_token = os.environ.get("HF_KEY", "").strip()
if not hub_token:
    hub_token = getpass.getpass("Hugging Face token (HF_KEY): ").strip()
if not hub_token:
    raise RuntimeError("Enter a Hugging Face token in HF_KEY or at the prompt.")

In [ ]:
from hamic_jax.checkpoint import HuggingFaceObjectStore

run_store = HuggingFaceObjectStore(HUB_REPO, private=True, token=hub_token)
data_store = HuggingFaceObjectStore(DATA_REPO, repo_type="dataset", create=False, token=hub_token)
print(f"Hub account {run_store.whoami()} · runs {HUB_REPO} · data {DATA_REPO}")

In [ ]:
from hamic_jax.lab import default_candidate, discover_training_configs

candidates, rejected = discover_training_configs(source_dir / "configs" / "train")
if not candidates:
    raise RuntimeError("No YAML run configuration is supported by the JAX backend.")
catalog = {candidate.name: candidate for candidate in candidates}
default = default_candidate(candidates).name
name = CONFIG_NAME.strip() or default
if name not in catalog:
    raise KeyError(f"{name!r} is not in {sorted(catalog)}")
candidate = catalog[name]
print(f"Configuration: {name}")
if rejected:
    print(f"Rejected: {rejected}")

In [ ]:
from hamic_jax.spec import read_yaml

config, preset = candidate.config, candidate.preset
raw_config = read_yaml(source_dir / candidate.name)
summary = [
    {"field": "run_id", "value": config.run_id},
    {"field": "config", "value": candidate.name},
    {"field": "architecture", "value": preset.architecture},
    {"field": "model", "value": preset.name},
    {"field": "parameters", "value": f"{preset.num_parameters:,}"},
    {"field": "objective", "value": config.objective},
    {"field": "masking", "value": config.masking},
    {"field": "seq_len", "value": config.seq_len},
    {"field": "tokens/step", "value": f"{config.tokens_per_step:,}"},
    {"field": "total tokens", "value": f"{config.total_tokens / 1e9:.3f} B"},
    {"field": "optimizer steps", "value": f"{config.total_steps:,}"},
    {"field": "tokenized", "value": config.tokenized},
]
for item in summary:
    print(f"{item['field']:>16}: {item['value']}")

In [ ]:
from hamic_jax.lab import fetch_corpus

config, corpus = fetch_corpus(config, data_store, work_root, config.run_id)
split_tokens = {name: value["tokens"] for name, value in corpus.manifest["splits"].items()}
epochs = config.total_tokens / split_tokens["train"]
print(f"Verified {corpus.root} · train {split_tokens['train']:,} tokens · {epochs:.2f} epochs")

In [ ]:
from tqdm.auto import tqdm

from hamic_jax.lab import run_bench, run_training_session

if MODE == "bench":
    learning_rate = float(BENCH_LR) if BENCH_LR.strip() else None
    warmup_ratio = float(BENCH_WARMUP) if BENCH_WARMUP.strip() else None
    with tqdm(total=int(BENCH_STEPS), desc=f"Bench {config.run_id}", unit="step") as bar:
        def progress(step, target, loss):
            bar.update(max(step - bar.n, 0))
            bar.set_postfix_str(f"loss {loss:.5f}")

        benchmark = run_bench(
            config,
            preset,
            corpus,
            work_root,
            steps=int(BENCH_STEPS),
            learning_rate=learning_rate,
            warmup_ratio=warmup_ratio,
            source_commit=commit,
            progress=progress,
            log=print,
        )
    result = benchmark.report
    print(
        f"Bench {benchmark.config.run_id} · lr {benchmark.config.learning_rate:g} "
        f"· warmup {benchmark.config.warmup_steps} steps · stop after {int(BENCH_STEPS)}"
    )
else:
    with tqdm(total=config.total_steps, desc=f"Train {config.run_id}", unit="step") as bar:
        def progress(step, target, loss):
            bar.update(max(step - bar.n, 0))
            bar.set_postfix_str(f"loss {loss:.5f}")

        result = run_training_session(
            config,
            preset,
            corpus,
            run_store,
            work_root / "runs" / config.run_id,
            source_commit=commit,
            raw_config=raw_config,
            progress=progress,
            log=print,
        )

In [ ]:
from IPython.display import display

from hamic_jax.plots import loss_curve

display(loss_curve(result, title=f"Loss · {result.run_id}"))

In [ ]:
from hamic_jax.lab import measure_throughput

throughput = measure_throughput(result, config, preset)
print(
    f"completed={result.completed} · step {result.step}/{config.total_steps} "
    f"· resumed from {result.resumed_from_step} · paused {result.paused}"
)
print(
    f"Session {result.seconds / 60:.1f} min · {throughput.tokens_per_second:,.0f} tokens/s "
    f"(~{throughput.tflops:.0f} TFLOP/s by 6·N·D) · about {throughput.hours_left:.1f} h left"
)
for record in result.metrics[-20:]:
    print({"step": record["step"], "loss": record["loss"], "tokens": record["tokens"]})

In [ ]:
evaluations = [
    {"step": record["step"], "eval_loss": record["eval_loss"]}
    for record in result.metrics
    if "eval_loss" in record
]
print("No evaluation in this session." if not evaluations else evaluations)